# Notebook 0 — Verificación del Entorno ⚙️
**TFG: Detección de Intrusiones a Gran Escala utilizando ML Distribuido y Big Data**

**Autor:** Eduardo Morillas Rodríguez

**Ejecutar UNA SOLA VEZ** al configurar el entorno. Después ir directo al NB1.

---
## 0.0 — Importar config.py

In [1]:
try:
	from config import *
	print("✅ config.py importado correctamente.")
	print(f"   BASE_PATH = {BASE_PATH}")
except ImportError as e:
	print(f"❌ ERROR: {e}")
	print("   Asegúrate de que config.py está en el mismo directorio.")
	raise

✅ config.py importado correctamente.
   BASE_PATH = /home/hpc/22231088student/tfg_ids


---
## 0.1 — Documentación del Nodo

### LORCA Computing Center — Universidad Europea de Madrid

| Recurso | Valor |
|---|---|
| **Nodo** | `eespcachcpro01` (grupo High Computing Research) |
| **CPU** | Intel Xeon Gold 6338 @ 2.0 GHz |
| **Cores** | 32 físicos / 64 hilos lógicos |
| **RAM** | 125 GB total (~100 GB disponibles) |
| **GPU** | NVIDIA RTX 3080 Ti — 12 GB GDDR6X |
| **CUDA** | 13.1 · Driver 590.48.01 |
| **Disco** | 20 GB cuota personal (SSD/NVMe, 300 GB compartidos) |
| **SLURM** | `low_intensity` (nodos 01-02), `high_intensity` (nodo 03) |
| **Acceso** | JupyterHub (10.151.52.x) + SSH + FortiClient VPN |
| **SO** | Ubuntu / Debian |

### Modo de ejecución
Spark se ejecuta en modo **`local[*]`** (64 hilos) dentro de un solo nodo.
La escalabilidad se analiza variando el número de hilos: `local[8]`, `local[16]`, `local[32]`, `local[64]`.

In [2]:
# === SCRIPT .SBATCH DE REFERENCIA ===


sbatch_template = """#!/bin/bash
#SBATCH --job-name=tfg_ids
#SBATCH --output=logs/%x_%j.out
#SBATCH --error=logs/%x_%j.err
#SBATCH --partition=low_intensity      	# Partición del nodo 01-02
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64             	# Todos los hilos del Xeon Gold 6338
#SBATCH --mem=100G                     	# ~100 GB de los 125 GB disponibles
#SBATCH --gres=gpu:1                   	# RTX 3080 Ti
#SBATCH --time=08:00:00                	# Tiempo máximo


# Activar entorno virtual
source ~/tfg_env/bin/activate


# Ejecutar notebook
python NB1_ingesta.py
"""


print("📋 Script .sbatch de referencia para SLURM:")
print(sbatch_template)

📋 Script .sbatch de referencia para SLURM:
#!/bin/bash
#SBATCH --job-name=tfg_ids
#SBATCH --output=logs/%x_%j.out
#SBATCH --error=logs/%x_%j.err
#SBATCH --partition=low_intensity      	# Partición del nodo 01-02
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64             	# Todos los hilos del Xeon Gold 6338
#SBATCH --mem=100G                     	# ~100 GB de los 125 GB disponibles
#SBATCH --gres=gpu:1                   	# RTX 3080 Ti
#SBATCH --time=08:00:00                	# Tiempo máximo


# Activar entorno virtual
source ~/tfg_env/bin/activate


# Ejecutar notebook
python NB1_ingesta.py



---
## 0.2 — Verificación de Dependencias Python

In [3]:
required = {
    "pyspark": 	"Motor de procesamiento distribuido",
    "matplotlib":  "Gráficos estáticos",
    "seaborn": 	"Gráficos estadísticos",
    "plotly":  	"Gráficos interactivos",
    "pandas":  	"Manipulación de datos (local)",
    "numpy":   	"Cálculo numérico",
    "sklearn": 	"ML (métricas, PCA local, t-SNE)",
    "scipy":   	"Tests estadísticos, QQ-plots",
    "statsmodels": "VIF (Variance Inflation Factor)",
    "missingno":   "Auditoría visual de datos faltantes",
}

print("=" * 60)
print("VERIFICACIÓN DE DEPENDENCIAS")
print("=" * 60)

all_ok = True
for package, desc in required.items():
    try:
        __import__(package)
        print(f"  ✅ {package:<15} — {desc}")
    except ImportError:
        print(f"  ❌ {package:<15} — NO encontrado. pip install {package}")
        all_ok = False

print(f"\n  Resultado: {'✅ Todas instaladas' if all_ok else '❌ Faltan dependencias'}")



VERIFICACIÓN DE DEPENDENCIAS
  ✅ pyspark         — Motor de procesamiento distribuido
  ✅ matplotlib      — Gráficos estáticos
  ✅ seaborn         — Gráficos estadísticos
  ✅ plotly          — Gráficos interactivos
  ✅ pandas          — Manipulación de datos (local)
  ✅ numpy           — Cálculo numérico
  ✅ sklearn         — ML (métricas, PCA local, t-SNE)
  ✅ scipy           — Tests estadísticos, QQ-plots
  ✅ statsmodels     — VIF (Variance Inflation Factor)
  ✅ missingno       — Auditoría visual de datos faltantes

  Resultado: ✅ Todas instaladas


---
## 0.3 — Verificación de GPU y CUDA

In [5]:
import subprocess


print("=" * 60)
print("VERIFICACIÓN DE GPU")
print("=" * 60)


gpu_ok = False
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print(result.stdout)
        gpu_ok = True
    else:
        print(f"  ⚠️ Error: {result.stderr}")
except FileNotFoundError:
    print("  ⚠️ nvidia-smi no encontrado.")
except subprocess.TimeoutExpired:
    print("  ⚠️ Timeout.")


print(f"\n  Esperado: NVIDIA RTX 3080 Ti · 12 GB · CUDA 13.1")

VERIFICACIÓN DE GPU
Sun May  3 21:25:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080 Ti     Off |   00000000:C3:00.0 Off |                  N/A |
|  0%   34C    P8             23W /  350W |       1MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

---
## 0.4 — Verificación de Espacio en Disco

In [7]:
import shutil

print("=" * 60)
print("ESPACIO EN DISCO")
print("=" * 60)

total, used, free = shutil.disk_usage(BASE_PATH)
free_gb = free / (1024**3)

print(f"  Directorio: {BASE_PATH}")
print(f"  Total:      {total / (1024**3):.2f} GB")
print(f"  Usado:      {used / (1024**3):.2f} GB")
print(f"  Libre:      {free_gb:.2f} GB")
print(f"  Tu cuota:   ~{DISK_QUOTA_GB} GB")

disk_ok = free_gb > 10
estado = "✅ OK" if disk_ok else "⚠️ Crítico (menos de 10GB libres)"
print(f"  Estado:     {estado}")

print(f"\n  📋 PLAN DE ESPACIO ({DISK_QUOTA_GB} GB cuota):")

col1_w = 34  # Ancho de la columna "Elemento"
col2_w = 12  # Ancho de la columna "Tamaño est."

# Borde superior
print(f"  ┌{'─' * (col1_w + 2)}┬{'─' * (col2_w + 2)}┐")
print(f"  │ {'Elemento':<{col1_w}} │ {'Tamaño est.':<{col2_w}} │")

# Separador interno
print(f"  ├{'─' * (col1_w + 2)}┼{'─' * (col2_w + 2)}┤")
print(f"  │ {'CSVs originales (temporal)':<{col1_w}} │ {'~6-8 GB':<{col2_w}} │")
print(f"  │ {'Parquet limpio (snappy)':<{col1_w}} │ {'~2-3 GB':<{col2_w}} │")
print(f"  │ {'Modelos guardados':<{col1_w}} │ {'~0.5 GB':<{col2_w}} │")
print(f"  │ {'Notebooks + gráficos':<{col1_w}} │ {'~0.5 GB':<{col2_w}} │")

# Separador interno para el total
print(f"  ├{'─' * (col1_w + 2)}┼{'─' * (col2_w + 2)}┤")
print(f"  │ {'TOTAL ESTIMADO':<{col1_w}} │ {'~10-12 GB':<{col2_w}} │")

# Borde inferior
print(f"  └{'─' * (col1_w + 2)}┴{'─' * (col2_w + 2)}┘")

print(f"\n  💡 Eliminar CSVs tras convertir a Parquet → libera ~6-8 GB.")

ESPACIO EN DISCO
  Directorio: /home/hpc/22231088student/tfg_ids
  Total:      299.85 GB
  Usado:      182.00 GB
  Libre:      117.85 GB
  Tu cuota:   ~20 GB
  Estado:     ✅ OK

  📋 PLAN DE ESPACIO (20 GB cuota):
  ┌────────────────────────────────────┬──────────────┐
  │ Elemento                           │ Tamaño est.  │
  ├────────────────────────────────────┼──────────────┤
  │ CSVs originales (temporal)         │ ~6-8 GB      │
  │ Parquet limpio (snappy)            │ ~2-3 GB      │
  │ Modelos guardados                  │ ~0.5 GB      │
  │ Notebooks + gráficos               │ ~0.5 GB      │
  ├────────────────────────────────────┼──────────────┤
  │ TOTAL ESTIMADO                     │ ~10-12 GB    │
  └────────────────────────────────────┴──────────────┘

  💡 Eliminar CSVs tras convertir a Parquet → libera ~6-8 GB.


---
## 0.5 — Test de SparkSession

In [8]:
print("=" * 60)
print("TEST DE SPARKSESSION")
print("=" * 60)


spark_ok = False
try:
	spark = get_spark_session("TFG_IDS_NB0_Test")


	print(f"  ✅ SparkSession creada.")
	print(f" 	App: 	{spark.sparkContext.appName}")
	print(f" 	Master:  {spark.sparkContext.master}")
	print(f" 	Version: {spark.version}")
	print(f" 	UI:  	http://localhost:4040")


	test_df = spark.createDataFrame([(1, "ok"), (2, "ok")], ["id", "status"])
	assert test_df.count() == 2
	print(f"\n  ✅ Test rápido: DataFrame creado y verificado.")
	spark_ok = True


	spark.stop()
	print(f"  ✅ SparkSession detenida.")


except Exception as e:
	print(f"  ❌ Error: {e}")
	print(f" 	Verificar: java -version")

TEST DE SPARKSESSION


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 21:27:19 WARN Utils: Your hostname, eespcachcpro01, resolves to a loopback address: 127.0.1.1; using 10.151.52.2 instead (on interface ens3f0)
26/05/03 21:27:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 21:27:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 21:27:20 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


  ✅ SparkSession creada.
 	App: 	TFG_IDS_NB0_Test
 	Master:  local[*]
 	Version: 4.1.1
 	UI:  	http://localhost:4040



  ✅ Test rápido: DataFrame creado y verificado.
  ✅ SparkSession detenida.


---
## 0.6 — Verificación de config.py

In [9]:
print("=" * 60)
print("VERIFICACIÓN DE config.py")
print("=" * 60)


print("\n📁 PATHS:")
for name, path in {
	"BASE_PATH": BASE_PATH,
	"DATA_RAW_PATH": DATA_RAW_PATH,
	"DATA_PARQUET_PATH": DATA_PARQUET_PATH,
	"DATA_CLEAN_PATH": DATA_CLEAN_PATH,
	"DATA_FINAL_PATH": DATA_FINAL_PATH,
	"MODELS_PATH": MODELS_PATH,
	"FIGURES_PATH": FIGURES_PATH,
	"ANNEX_PATH": ANNEX_PATH,
	"LOGS_PATH": LOGS_PATH,
}.items():
	exists = "✅" if os.path.exists(path) else "❌"
	print(f"  {exists} {name:<20} = {path}")


print("\n🔢 VARIABLES:")
for name, val in {
	"SEED": SEED,
	"TRAIN_RATIO": TRAIN_RATIO,
	"CORRELATION_THRESHOLD": CORRELATION_THRESHOLD,
	"WARNING_CORRUPTION_PCT": WARNING_CORRUPTION_PCT,
	"NODE_NAME": NODE_NAME,
	"CPU_MODEL": CPU_MODEL,
	"LOGICAL_THREADS": LOGICAL_THREADS,
	"RAM_TOTAL_GB": RAM_TOTAL_GB,
	"GPU_MODEL": GPU_MODEL,
}.items():
	print(f"  ✅ {name:<25} = {val}")


print(f"\n🏷️  VALID_LABELS:   {len(VALID_LABELS)} clases")
print(f"🎨 COLOR_PALETTE:  {len(COLOR_PALETTE)} colores")


labels_sin_color = [l for l in VALID_LABELS if l not in COLOR_PALETTE]
print(f"  {'⚠️ Sin color: ' + str(labels_sin_color) if labels_sin_color else '✅ Todos los labels tienen color.'}")


print(f"\n🔧 FUNCIONES:")
for fn in ["save_figure", "save_annex_figure", "get_size_mb", "print_dataframe_info", "get_spark_session"]:
	print(f"  {'✅' if fn in dir() else '❌'} {fn}()")

VERIFICACIÓN DE config.py

📁 PATHS:
  ✅ BASE_PATH            = /home/hpc/22231088student/tfg_ids
  ✅ DATA_RAW_PATH        = /home/hpc/22231088student/tfg_ids/data/raw
  ✅ DATA_PARQUET_PATH    = /home/hpc/22231088student/tfg_ids/data/parquet/cse_cic_ids_2018
  ✅ DATA_CLEAN_PATH      = /home/hpc/22231088student/tfg_ids/data/clean
  ✅ DATA_FINAL_PATH      = /home/hpc/22231088student/tfg_ids/data/final
  ✅ MODELS_PATH          = /home/hpc/22231088student/tfg_ids/models
  ✅ FIGURES_PATH         = /home/hpc/22231088student/tfg_ids/figures
  ✅ ANNEX_PATH           = /home/hpc/22231088student/tfg_ids/figures/anexo
  ✅ LOGS_PATH            = /home/hpc/22231088student/tfg_ids/logs

🔢 VARIABLES:
  ✅ SEED                      = 42
  ✅ TRAIN_RATIO               = 0.8
  ✅ CORRELATION_THRESHOLD     = 0.95
  ✅ WARNING_CORRUPTION_PCT    = 10
  ✅ NODE_NAME                 = eespcachcpro01
  ✅ CPU_MODEL                 = Intel Xeon Gold 6338
  ✅ LOGICAL_THREADS           = 64
  ✅ RAM_TOTAL_GB            

In [10]:
# === TEST save_figure() ===
print("\n🧪 Test de save_figure():")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["Test A", "Test B", "Test C"], [3, 7, 5], color=["#2ecc71", "#e74c3c", "#3498db"])
ax.set_title("Gráfico de prueba — save_figure()")
save_figure(fig, "00_test_verificacion")

test_fig_path = os.path.join(FIGURES_PATH, "00_test_verificacion.png")
if os.path.exists(test_fig_path):
	print(f"  ✅ Archivo creado: {os.path.getsize(test_fig_path)/1024:.1f} KB")
else:
	print(f"  ❌ No se creó. Revisar permisos.")


🧪 Test de save_figure():
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/00_test_verificacion.png
  ✅ Archivo creado: 21.9 KB


In [11]:
# === TEST save_annex_figure() ===
print("\n🧪 Test de save_annex_figure():")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["Test X", "Test Y"], [4, 6], color=["#f39c12", "#9b59b6"])
ax.set_title("Gráfico de prueba — save_annex_figure()")
save_annex_figure(fig, "00_test_anexo")

test_annex_path = os.path.join(ANNEX_PATH, "00_test_anexo.png")
if os.path.exists(test_annex_path):
	print(f"  ✅ Archivo creado: {os.path.getsize(test_annex_path)/1024:.1f} KB")
else:
	print(f"  ❌ No se creó. Revisar permisos de {ANNEX_PATH}")


🧪 Test de save_annex_figure():
  💾 Figura guardada (Anexo): /home/hpc/22231088student/tfg_ids/figures/anexo/00_test_anexo.png
  ✅ Archivo creado: 22.9 KB


---
## 0.7 — Estructura de Directorios

In [12]:
print("=" * 60)
print("ESTRUCTURA DE DIRECTORIOS")
print("=" * 60)
print(f"""
  {BASE_PATH}/
  ├── data/
  │   ├── raw/          	← CSVs originales (eliminar tras Parquet)
  │   ├── parquet/      	← Datos tras ingesta (NB1)
  │   ├── clean/        	← Preprocesados (NB3)
  │   └── final/        	← Listos para modelado (NB4)
  ├── models/           	← Modelos entrenados (NB5)
  ├── figures/          	← Gráficos del cuerpo del TFG
  │   └── anexo/        	← Gráficos complementarios (Anexo)
  ├── logs/             	← Logs SLURM/Spark
  ├── spark_tmp/        	← Temporal de Spark
  ├── config.py         	← ⭐ Configuración compartida
  └── NB0..NB7          	← Notebooks
""")


all_dirs_ok = all(os.path.isdir(p) for p in ALL_PATHS)
print(f"  {'✅ Todos los directorios existen.' if all_dirs_ok else '❌ Algún directorio falta.'}")

ESTRUCTURA DE DIRECTORIOS

  /home/hpc/22231088student/tfg_ids/
  ├── data/
  │   ├── raw/          	← CSVs originales (eliminar tras Parquet)
  │   ├── parquet/      	← Datos tras ingesta (NB1)
  │   ├── clean/        	← Preprocesados (NB3)
  │   └── final/        	← Listos para modelado (NB4)
  ├── models/           	← Modelos entrenados (NB5)
  ├── figures/          	← Gráficos del cuerpo del TFG
  │   └── anexo/        	← Gráficos complementarios (Anexo)
  ├── logs/             	← Logs SLURM/Spark
  ├── spark_tmp/        	← Temporal de Spark
  ├── config.py         	← ⭐ Configuración compartida
  └── NB0..NB7          	← Notebooks

  ✅ Todos los directorios existen.


---
## ✅ Checklist Final

In [14]:
print("\n" + "=" * 60)
print("📋 CHECKLIST FINAL")
print("=" * 60)


checks = {
    "config.py importado":      	True,
    "Dependencias Python":      	all_ok,
    f"GPU ({GPU_MODEL})":       	gpu_ok,
    f"Espacio disco (>{DISK_QUOTA_GB//2} GB libres)": disk_ok,
    "SparkSession (local[*])":  	spark_ok,
    "Directorios creados":      	all_dirs_ok,
    "save_figure() funciona":   	os.path.exists(test_fig_path),
    "save_annex_figure() funciona": os.path.exists(test_annex_path),
}


all_passed = True
for check, passed in checks.items():
    status = "✅" if passed else "⚠️"
    if not passed:
        all_passed = False
    print(f"  {status} {check}")


print(f"\n{'=' * 60}")
if all_passed:
    print("  🎉 TODO OK — Entorno listo.")
else:
    print("  ⚠️  Elementos pendientes arriba.")
print("  ➡️  Siguiente: NB1 — Ingesta de Datos")
print(f"{'=' * 60}")


📋 CHECKLIST FINAL
  ✅ config.py importado
  ✅ Dependencias Python
  ✅ GPU (NVIDIA RTX 3080 Ti)
  ✅ Espacio disco (>10 GB libres)
  ✅ SparkSession (local[*])
  ✅ Directorios creados
  ✅ save_figure() funciona
  ✅ save_annex_figure() funciona

  🎉 TODO OK — Entorno listo.
  ➡️  Siguiente: NB1 — Ingesta de Datos
